# Phase 0: Setup
Setting up FastF1 where we will be gathering our data for the project

In [1]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

fastf1.Cache.enable_cache("../cache/")

# Phase 1: EDA: Bahrain Grand Prix 2023 analysis


We are starting by analyzing one race, preferably in a race where the circuit is pretty stable with overtaking opportunities, balanced pace etc. One of the choices is the Bahrain circuit, it's special because it's where tyre degradation appears to be and there are lots of overtakings, one of the reasons why it's the pre-season testing circuit so it's where teams actually test their cars before the season begins.

We load and analyze the race data

In [2]:
bahrain2023_race = fastf1.get_session(2023, "Bahrain", "R")
bahrain2023_race.load()
laps = bahrain2023_race.laps

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [3]:
print(laps.shape)
print(laps.columns)
print(laps.dtypes)
print(laps.head())


(1056, 31)
Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')
Time                  timedelta64[ns]
Driver                         object
DriverNumber                   object
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedel

## Feature Engineering and cleaning

Before going forward we have to clean our data, there are laps that don't count, either in/out laps, laps under safety car, invalid laps etc and we need to fixate on one driver to actually have a consistent experiment piece

In [5]:
laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()
laps["Stint"] = laps["Stint"].astype(int)
clean_laps = laps[laps["IsAccurate"] == True].copy()
drivers = clean_laps["Driver"].unique()
print(f" Missed values: {clean_laps.isna().sum()}")
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    print("Driver: ", d)
    print(ds)
    for s in ds:
        print(f"stint {s}: ", clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)].shape[0])

 Missed values: Time                    0
Driver                  0
DriverNumber            0
LapTime                 0
LapNumber               0
Stint                   0
PitOutTime            914
PitInTime             914
Sector1Time             0
Sector2Time             0
Sector3Time             0
Sector1SessionTime      4
Sector2SessionTime      0
Sector3SessionTime      0
SpeedI1                 0
SpeedI2                 0
SpeedFL                 0
SpeedST                 0
IsPersonalBest          0
Compound                0
TyreLife                0
FreshTyre               0
Team                    0
LapStartTime            0
LapStartDate            0
TrackStatus             0
Position                0
Deleted                 0
DeletedReason           0
FastF1Generated         0
IsAccurate              0
LapTimeSeconds          0
dtype: int64
Driver:  VER
[1 2 3]
stint 1:  12
stint 2:  20
stint 3:  18
Driver:  GAS
[1 2 3 4]
stint 1:  7
stint 2:  14
stint 3:  13
stint 4:  16
Drive

In [ ]:
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    for s in ds:
        driver_stint = clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)]
        compound = driver_stint["Compound"].iloc[0]
        plt.xlabel("Tyre Age")
        plt.ylabel("Lap Time")
        plt.title(f"{d} Stint {s} Compound: {compound} Bahrain 2023 laptime evolution by tyre age (Lower is faster)")
        plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
        plt.savefig(f"../figs/bahrain2023_{d}_stint{s}_{compound}_tyredeg_evo.png")
        plt.cla()
plt.close()

We need to analyze the fuel effect in the results, but before that we need to get the coefficient of how many seconds we save per kg of fuel burned, we do the analysis on one driver:

In [ ]:
START_FUEL = 109.0
FUEL_BURN = START_FUEL/57
sec_per_kg_fuel = []
for d in drivers:
    driver_laps = clean_laps[clean_laps["Driver"] == d].copy()
    if driver_laps["LapNumber"].iloc[-1] < 50.0 :
        continue
    y = driver_laps[['LapTimeSeconds']]
    driver_laps["FuelMass"] = START_FUEL - (FUEL_BURN * driver_laps["LapNumber"]) + 1
    X = driver_laps[["TyreLife", "FuelMass"]]
    laptime_predictor = LinearRegression()
    laptime_predictor.fit(X, y)
    print(f"Driver: {d}, deg_rate = {laptime_predictor.coef_[0][0]}, fuel_coef ={laptime_predictor.coef_[0][1]}")
    sec_per_kg_fuel.append(laptime_predictor.coef_[0][1])
gamma = np.mean(sec_per_kg_fuel)
per_lap_gain = gamma * FUEL_BURN
race_cum_gain = per_lap_gain * 57
print(f"Mean of seconds earned per kg: {gamma}")
print(f"Per-lap gain in seconds: {per_lap_gain}")
print(f"Race gain in seconds: {race_cum_gain}")


In [ ]:
clean_laps["FuelMass"] = START_FUEL - (FUEL_BURN * clean_laps["LapNumber"]) + 1
clean_laps["FuelCorrectedLapTime"] = clean_laps["LapTimeSeconds"] - gamma * clean_laps["FuelMass"]
print(clean_laps[["LapNumber", "Driver", "LapTimeSeconds", "FuelMass", "FuelCorrectedLapTime"]])
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    for s in ds:
        driver_stint = clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)]
        compound = driver_stint["Compound"].iloc[0]
        plt.xlabel("Tyre Age")
        plt.ylabel("Lap Time")
        plt.title(f"{d} Stint {s} Compound: {compound} Bahrain 2023 laptime evolution by tyre age (Fuel-corrected) (Lower is faster)")
        plt.scatter(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['FuelCorrectedLapTime']))
        plt.savefig(f"../figs/bahrain2023_{d}_stint{s}_{compound}_tyredeg_evo_fuel_corrected.png")
        plt.cla()
plt.close()

Wrapping everything to one function to automatically extract the results per race returning the final dataset

In [8]:
def race_analysis(gp, year):
    race = fastf1.get_session(year, gp, "R")
    race.load()
    laps = race.laps
    laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()
    laps["Stint"] = laps["Stint"].astype(int)
    clean_laps = laps[laps["IsAccurate"] == True].copy()
    drivers = clean_laps["Driver"].unique()
    START_FUEL = 109.0
    RACE_LAPS = clean_laps["LapNumber"].max()
    FUEL_BURN = START_FUEL/RACE_LAPS
    print(f"Laps: {RACE_LAPS}")
    sec_per_kg_fuel = []
    for d in drivers:
        driver_laps = clean_laps[clean_laps["Driver"] == d].copy()
        if driver_laps["LapNumber"].iloc[-1] < clean_laps["LapNumber"].max() - 5.0 :
            continue
        y = driver_laps[['LapTimeSeconds']]
        driver_laps["FuelMass"] = START_FUEL - (FUEL_BURN * driver_laps["LapNumber"]) + 1
        X = driver_laps[["TyreLife", "FuelMass"]]
        laptime_predictor = LinearRegression()
        laptime_predictor.fit(X, y)
        print(f"Driver: {d}, deg_rate = {laptime_predictor.coef_[0][0]}, fuel_coef ={laptime_predictor.coef_[0][1]}")
        sec_per_kg_fuel.append(laptime_predictor.coef_[0][1])
    gamma = np.mean(sec_per_kg_fuel)
    clean_laps["FuelMass"] = START_FUEL - (FUEL_BURN * clean_laps["LapNumber"]) + 1
    clean_laps["FuelCorrectedLapTime"] = clean_laps["LapTimeSeconds"] - gamma * clean_laps["FuelMass"]
    print(clean_laps[["LapNumber", "Driver", "LapTimeSeconds", "FuelMass", "FuelCorrectedLapTime"]])
    
    for d in drivers:
        ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
        for s in ds:
            driver_stint = clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)]
            compound = driver_stint["Compound"].iloc[0]
            plt.xlabel("Tyre Age")
            plt.ylabel("Lap Time")
            plt.title(f"{d} Stint {s} Compound: {compound} {gp} {year} laptime evolution by tyre age (Lower is faster)")
            plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
            plt.savefig(f"../figs/{gp}{year}_{d}_stint{s}_{compound}_tyredeg_evo.png")
            plt.cla()
    plt.close()

    return clean_laps, gamma

race_data = race_analysis("Miami", 2024)
print(race_data[0])
print(f"Gamma: {race_data[1]}")

core           INFO 	Loading data for Miami Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cach

Laps: 57.0
Driver: VER, deg_rate = 0.03079868124621089, fuel_coef =0.02780291670444947
Driver: GAS, deg_rate = 0.01360326035104375, fuel_coef =0.02367755967217189
Driver: PER, deg_rate = 0.004795067660458852, fuel_coef =0.025772082120717972
Driver: ALO, deg_rate = 0.024135948285333286, fuel_coef =0.032887091613266535
Driver: LEC, deg_rate = 0.027961501405798353, fuel_coef =0.03057877192046853
Driver: STR, deg_rate = -0.011189915391207467, fuel_coef =0.018907725858455793
Driver: MAG, deg_rate = 0.06228169375063772, fuel_coef =0.03812806365157197
Driver: TSU, deg_rate = 0.015907353421673587, fuel_coef =0.03127690005179183
Driver: ALB, deg_rate = 0.03329644709194948, fuel_coef =0.02933162501616366
Driver: ZHO, deg_rate = 0.0031524782174623465, fuel_coef =0.02207985055730322
Driver: HUL, deg_rate = 0.03419389854230272, fuel_coef =0.029220068403205186
Driver: RIC, deg_rate = -0.0038384730125407564, fuel_coef =0.02328354731275935
Driver: OCO, deg_rate = 0.020839679518106675, fuel_coef =0.027

Seems a decent baseline, let's continue with EDA

# Phase 2: Feature Engineering and Preprocessing